# 5. Numerical ODE Solvers

Solve initial value problems $y' = f(t, y),\; y(t_0) = y_0$. This notebook covers:
- **Euler's method** (explicit, order 1)
- **Classical Runge-Kutta (RK4)** (order 4)
- Comparison on standard ODEs
- Global error analysis and stability

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

## 5.1 Euler's Method

The simplest ODE solver:
$$y_{n+1} = y_n + h \cdot f(t_n, y_n)$$

Local truncation error: $O(h^2)$. Global error: $O(h)$.

In [ ]:
def euler(f, t_span, y0, h):
    t0, tf = t_span
    t = np.arange(t0, tf + h/2, h)
    y = np.zeros(len(t))
    y[0] = y0
    for i in range(len(t) - 1):
        y[i+1] = y[i] + h * f(t[i], y[i])
    return t, y

# Test: y' = -2y, y(0) = 1  -->  exact solution: y(t) = e^{-2t}
f_exp = lambda t, y: -2 * y
exact_exp = lambda t: np.exp(-2 * t)

t_e, y_e = euler(f_exp, (0, 3), 1.0, 0.1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
t_fine = np.linspace(0, 3, 200)
axes[0].plot(t_fine, exact_exp(t_fine), 'k-', lw=2, label='Exact')
axes[0].plot(t_e, y_e, 'ro--', markersize=4, label='Euler (h=0.1)')
axes[0].set_xlabel('t')
axes[0].set_ylabel('y(t)')
axes[0].set_title("Euler's Method")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_e, np.abs(y_e - exact_exp(t_e)), 'r-o', markersize=3)
axes[1].set_xlabel('t')
axes[1].set_ylabel('|error|')
axes[1].set_title('Euler Error')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5.2 Classical Runge-Kutta (RK4)

$$y_{n+1} = y_n + \frac{h}{6}(k_1 + 2k_2 + 2k_3 + k_4)$$

where:
- $k_1 = f(t_n, y_n)$
- $k_2 = f(t_n + h/2, y_n + hk_1/2)$
- $k_3 = f(t_n + h/2, y_n + hk_2/2)$
- $k_4 = f(t_n + h, y_n + hk_3)$

Global error: $O(h^4)$.

In [ ]:
def rk4(f, t_span, y0, h):
    t0, tf = t_span
    t = np.arange(t0, tf + h/2, h)
    y = np.zeros(len(t))
    y[0] = y0
    for i in range(len(t) - 1):
        k1 = f(t[i], y[i])
        k2 = f(t[i] + h/2, y[i] + h*k1/2)
        k3 = f(t[i] + h/2, y[i] + h*k2/2)
        k4 = f(t[i] + h, y[i] + h*k3)
        y[i+1] = y[i] + h/6 * (k1 + 2*k2 + 2*k3 + k4)
    return t, y

t_rk, y_rk = rk4(f_exp, (0, 3), 1.0, 0.1)
print(f"Max error Euler (h=0.1): {np.max(np.abs(y_e - exact_exp(t_e))):.6e}")
print(f"Max error RK4   (h=0.1): {np.max(np.abs(y_rk - exact_exp(t_rk))):.6e}")

## 5.3 A More Interesting ODE: Logistic Equation

$$y' = ry\left(1 - \frac{y}{K}\right), \quad y(0) = y_0$$

Exact solution: $y(t) = \frac{K}{1 + \left(\frac{K}{y_0} - 1\right)e^{-rt}}$

In [ ]:
r, K, y0_log = 1.0, 100.0, 5.0
f_log = lambda t, y: r * y * (1 - y / K)
exact_log = lambda t: K / (1 + (K / y0_log - 1) * np.exp(-r * t))

t_e_l, y_e_l = euler(f_log, (0, 10), y0_log, 0.5)
t_rk_l, y_rk_l = rk4(f_log, (0, 10), y0_log, 0.5)

t_fine = np.linspace(0, 10, 300)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(t_fine, exact_log(t_fine), 'k-', lw=2, label='Exact')
ax.plot(t_e_l, y_e_l, 'ro--', markersize=4, label='Euler (h=0.5)')
ax.plot(t_rk_l, y_rk_l, 'bs-', markersize=4, label='RK4 (h=0.5)')
ax.set_xlabel('t')
ax.set_ylabel('y(t)')
ax.set_title('Logistic Equation: Euler vs RK4')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5.4 Global Error vs Step Size

In [ ]:
step_sizes = [0.5, 0.25, 0.1, 0.05, 0.025, 0.01]
errors_euler = []
errors_rk4 = []

for h in step_sizes:
    t_e, y_e = euler(f_exp, (0, 3), 1.0, h)
    t_r, y_r = rk4(f_exp, (0, 3), 1.0, h)
    errors_euler.append(np.max(np.abs(y_e - exact_exp(t_e))))
    errors_rk4.append(np.max(np.abs(y_r - exact_exp(t_r))))

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(step_sizes, errors_euler, 'ro-', markersize=6, label='Euler')
ax.loglog(step_sizes, errors_rk4, 'bs-', markersize=6, label='RK4')
# Reference lines
h_ref = np.array(step_sizes)
ax.loglog(h_ref, 0.5 * h_ref, 'r--', alpha=0.3, label='$O(h)$')
ax.loglog(h_ref, 0.01 * h_ref**4, 'b--', alpha=0.3, label='$O(h^4)$')
ax.set_xlabel('Step size h')
ax.set_ylabel('Max global error')
ax.set_title('Error vs Step Size')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Key Takeaways

| Method | Order | Cost per step | Accuracy |
|--------|-------|--------------|----------|
| Euler | 1 | 1 evaluation | Low |
| RK4 | 4 | 4 evaluations | High |

- **Euler** is simple but requires very small $h$ for accuracy
- **RK4** is the workhorse of ODE solving: excellent accuracy-to-cost ratio
- To halve the error: Euler needs $h/2$ (2x cost), RK4 needs $h/2^{1/4} \approx 0.84h$ (much cheaper)
- In practice, use **adaptive** step-size methods (e.g., `scipy.integrate.solve_ivp`)